## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [1]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [2]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [3]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [4]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [5]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [6]:
# Write collection_bbox(features) and apply to all four LOD files

import json
from pathlib import Path

lod_dir = Path("../../data/lod")

lod_files = {
    "coarse": "railroads_coarse.geojson",
    "medium": "railroads_medium.geojson",
    "fine": "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]

    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]

    return [min(lons), min(lats), max(lons), max(lats)]


def collection_bbox(features):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a list of GeoJSON features.
    """
    all_bboxes = []

    for feature in features:
        bbox = feature_bbox(feature)
        all_bboxes.append(bbox)

    lon_mins = [bbox[0] for bbox in all_bboxes]
    lat_mins = [bbox[1] for bbox in all_bboxes]
    lon_maxs = [bbox[2] for bbox in all_bboxes]
    lat_maxs = [bbox[3] for bbox in all_bboxes]

    return [
        min(lon_mins),
        min(lat_mins),
        max(lon_maxs),
        max(lat_maxs),
    ]


print(f"{'Level':<12} {'BBox'}")
print("-" * 70)

for level, filename in lod_files.items():
    path = lod_dir / filename

    with open(path) as f:
        fc = json.load(f)

    bbox = collection_bbox(fc["features"])

    print(f"{level:<12} {bbox}")

Level        BBox
----------------------------------------------------------------------
coarse       [-123.014722, -41.475186, 150.961667, 60.976516]
medium       [-150.112222, -51.894722, 179.357778, 69.604375]
fine         [-150.112222, -51.894722, 179.357778, 69.604375]
extra_fine   [-150.112222, -51.895278, 179.357778, 69.604375]


 The LOD files should cover mostly the same global geographic extent.
 However, the coarse file may have a slightly different extent because it uses a scalerank <= 4 filter, so it does not include every original feature.
 Medium, fine, and extra_fine should be closer to each other because they keep all features and mainly differ by geometry simplification.

## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [7]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson

fine_path = lod_dir / "railroads_fine.geojson"

with open(fine_path) as f:
    fine = json.load(f)

feature_areas = []

for i, feature in enumerate(fine["features"]):
    bbox = feature_bbox(feature)

    lon_min, lat_min, lon_max, lat_max = bbox

    width = lon_max - lon_min
    height = lat_max - lat_min
    area = width * height

    category = feature["properties"].get("category")

    feature_areas.append((area, i, bbox, category, feature))

top_5 = sorted(feature_areas, key=lambda x: x[0], reverse=True)[:5]

print("Top 5 features by bounding box area:")
print()

for area, index, bbox, category, feature in top_5:
    print("Feature index:", index)
    print("BBox area:", area)
    print("BBox:", bbox)
    print("Category:", category)
    print("Coordinate count:", len(feature["geometry"]["coordinates"]))
    print("-" * 40)

Top 5 features by bounding box area:

Feature index: 25411
BBox area: 29.312261184095934
BBox: [90.609351, 29.645107, 94.942815, 36.409271]
Category: 0
Coordinate count: 78
----------------------------------------
Feature index: 25409
BBox area: 18.64874534544004
BBox: [132.255704, -23.549819, 134.323448, -14.530934]
Category: 0
Coordinate count: 19
----------------------------------------
Feature index: 24238
BBox area: 17.704466501512
BBox: [-64.094167, -26.192499, -58.156944, -23.210555]
Category: 3
Coordinate count: 12
----------------------------------------
Feature index: 4803
BBox area: 17.386554647146994
BBox: [14.021914, 54.802728, 20.000001, 57.711109]
Category: 2
Coordinate count: 7
----------------------------------------
Feature index: 23589
BBox area: 12.354699164443995
BBox: [-49.137778, -5.491666, -44.372222, -2.899167]
Category: 2
Coordinate count: 29
----------------------------------------


The results make sense if the largest bounding boxes belong to railroad features that stretch long distances across countries or regions.

A large bbox area does not always mean the railroad itself fills that whole rectangle. It only means the feature reaches far enough east/west and
north/south to create a large enclosing box.

## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

Two railroad features could have identical bounding boxes if they start and
end near the same corners of the same rectangle but follow different paths.

For example, one railroad could curve across the top of the rectangle while
another curves across the bottom, but both still share the same lon_min,lat_min, lon_max, and lat_max.

This does not break the culling system. Bounding boxes are only a quick first
test for visibility. They may include a few extra features, but they should not accidentally remove visible features.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.